# Planet Motions and the Solar System


### Learning Goals
This notebook introduces the main concepts leading up to retrograde motion of the planets.

### Contents

Some description

### Setup

Let's load some python packages and code. The cells below contain python code that is labelled as follows:

- `SUPPORTING CODE`: You can ignore the lines of code in these cells but you **must** execute them.
- `ANIMATION`: Serves only to produce an animation. You can skip executing these cells if you are going quickly through the notebook, or if you are focusing on a specific exercise.
- `MAIN CODE`: Crucial code for producing subsequent results and plots. You will have to edit parts of these cells to carry out the exercises.

In [ ]:
### SUPPORTING CODE

%pip install skyfield

In [ ]:
### SUPPORTING CODE

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

from skyfield.api import load
from skyfield.data import hipparcos, stellarium, mpc
from skyfield.framelib import ecliptic_frame

from IPython.display import display,HTML
plt.rcParams['figure.dpi'] = 100  # Change to 200, 300, etc. for higher quality
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

In [ ]:
### SUPPORTING CODE

# Load Skyfield objects (needed later)
ts = load.timescale()
eph = load('de421.bsp')

# The Hipparcos mission provides our star catalog.
with load.open(hipparcos.URL) as f:
    stars = hipparcos.load_dataframe(f)

# And the constellation outlines come from Stellarium.
# We make a list of the stars at which each edge stars, and the star at which each edge ends.
url = ('https://raw.githubusercontent.com/Stellarium/stellarium/'
       'eb47095a9282cf6b981f6e37fe1ea3a3ae0fd167'
       '/skycultures/modern_st/constellationship.fab')

with load.open(url) as f:
    constellations = stellarium.parse_constellations(f)

In [ ]:
### SUPPORTING CODE

class planet():
    
    # Matching JPL names (left) to our planet names (right) for fetching observations
    jpl_planet_names = {
        'mercury': 'mercury',
        'venus': 'venus',
        'earth': 'earth',
        'mars': 'mars',
        'jupiter': 'jupiter barycenter',
        'saturn': 'saturn barycenter',
        'uranus': 'uranus barycenter',
        'neptune': 'neptune barycenter'
    }

    
    def __init__(self,name,period,eccentricity,semimajor,inclination,obliquity,omega_lan,omega_peri,mean_anomaly):
        self.name = name
        self.period = period  # in units of Earth years
        self.e = eccentricity # dimensionless (0 means circle)
        self.a = semimajor    # in units of AU
        self.i = inclination  # in units of degrees w.r.t. the Earth's orbital plane
        self.o = obliquity    # in units of degrees w.r.t. to the orbital plane of the planet
        self.omega_lan = omega_lan # in units of degrees (longitude of ascending node)
        self.omega_peri = omega_peri # in units of degrees (argument of periapsis)
        self.M0 = mean_anomaly # degrees from periapsis at J2000.0
        
        self.model_track_x = []    # perifocal in AU
        self.model_track_y = []    # perifocal in AU
        self.model_track_x_ec = [] # ecliptic in AU
        self.model_track_y_ec = [] # ecliptic in AU
        self.model_track_z_ec = [] # ecliptic in AU
        self.model_track_ra = []   # equatorial with respect to another planet
        self.model_track_dec = []  # equatorial with respect to another planet

        self.obs_t = []    # observed time in years
        self.obs_x_ec = [] # observed ecliptic in AU
        self.obs_y_ec = [] # observed ecliptic in AU
        self.obs_z_ec = [] # observed ecliptic in AU
        self.obs_ra = []   # equatorial with respect to another planet
        self.obs_dec = []  # equatorial with respect to another planet
        
        
        if name == "sun":
            self.color = "gold"
            self.marker = (8, 2, 0)
        elif self.name == "mercury":
            self.color = "silver"
            self.marker = 'o'
        elif self.name == "venus":
            self.color = "pink"
            self.marker = 'o'
        elif self.name == "earth":
            self.color = "royalblue"
            self.marker = 'o'
        elif self.name == "mars":
            self.color = "red"
            self.marker = 'o'
        elif self.name == "jupiter":
            self.color = "orange"
            self.marker = 'o'
        elif self.name == "saturn":
            self.color = "yellow"
            self.marker = 'o'
        elif self.name == "uranus":
            self.color = "cyan"
            self.marker = 'o'
        elif self.name == "neptune":
            self.color = "mediumblue"
            self.marker = 'o'
        else:
            self.color = "black"
            self.marker = 'o'

    ### KEPLERIAN MODEL
    def solve_kepler(self,M, e):
        E = M.copy()
        for _ in range(5):
            E = E - (E - e * np.sin(E) - M) / (1 - e * np.cos(E))
        return E

    def get_keplerian_track(self,t):
        # Get x,y of the planet in perifocal coordinates
        N_planet = 2 * np.pi / self.period
        M = np.radians(self.M0) + N_planet * (t-2000) # mean anomaly
        E = self.solve_kepler(M,self.e) # eccentric anomaly

        # Find the true anomaly 
        cos_nu = (np.cos(E) - self.e)/(1.0 - self.e*np.cos(E))
        sin_nu = np.sqrt(1 - self.e**2)*np.sin(E)/(1.0 - self.e*np.cos(E))
        #nu = np.atan2(sin_nu,cos_nu) # true anomaly
                
        r = self.a*(1.0-self.e**2)/(1.0+self.e*cos_nu)
        x = r*cos_nu
        y = r*sin_nu
        
        #x = self.a * (np.cos(E) - self.e)
        #y = self.a * np.sqrt(1 - self.e**2) * np.sin(E)
        self.model_track_x = x
        self.model_track_y = y


    def get_circular_track(self,t):
        # We can ignore the eccentric anomaly, the mean anomaly is the true anomaly
        # We are already in "perifocal" coordinates (the circle has only one focus, its center)        
        N_planet = 2 * np.pi / self.period
        M = np.radians(self.M0) + N_planet * (t-2000) # mean anomaly
        x = self.a*np.cos(M)
        y = self.a*np.sin(M)
        self.model_track_x = x
        self.model_track_y = y
        

    def convert_to_ecliptic(self):
        # Convert from perifocal to ecliptic coordinates
        i = np.radians(self.i)
        Wl = np.radians(self.omega_lan) # longitude of ascending node
        wp = np.radians(self.omega_peri) # argument of periapsis (not the same as longitude of periapsis)
        self.model_track_x_ec = self.model_track_x * (np.cos(Wl)*np.cos(wp) - np.sin(Wl)*np.sin(wp)*np.cos(i)) - self.model_track_y * (np.cos(Wl)*np.sin(wp) + np.sin(Wl)*np.cos(wp)*np.cos(i))
        self.model_track_y_ec = self.model_track_x * (np.sin(Wl)*np.cos(wp) + np.cos(Wl)*np.sin(wp)*np.cos(i)) - self.model_track_y * (np.sin(Wl)*np.sin(wp) - np.cos(Wl)*np.cos(wp)*np.cos(i))
        self.model_track_z_ec = self.model_track_x * (np.sin(wp)*np.sin(i)) + self.model_track_y * (np.cos(wp)*np.sin(i))
    

    def convert_to_eq(self,planet_ref,flag='model'):
        # We need to transform the planet's track from the ecliptic plane to the reference planet's equatorial frame
        
        # Get the vector to the reference planet (difference of position vectors)
        if flag == 'model':
            dx = self.model_track_x_ec - planet_ref.model_track_x_ec
            dy = self.model_track_y_ec - planet_ref.model_track_y_ec
            dz = self.model_track_z_ec - planet_ref.model_track_z_ec
        else:
            dx = self.obs_x_ec - planet_ref.model_track_x_ec
            dy = self.obs_y_ec - planet_ref.model_track_y_ec
            dz = self.obs_z_ec - planet_ref.model_track_z_ec
            
        # Convert this distance in equatorial coordinates of the reference planet
        ob = np.radians(planet_ref.o)
        dx_eq = dx
        dy_eq = dy * np.cos(ob) - dz * np.sin(ob)
        dz_eq = dy * np.sin(ob) + dz * np.cos(ob)
        dist = np.sqrt(dx_eq**2 + dy_eq**2 + dz_eq**2)
        dec = np.degrees( np.arcsin(dz_eq/dist) )
        ra = np.degrees( np.mod(np.arctan2(dy_eq,dx_eq),2*np.pi)  )

        if flag == 'model':
            self.model_track_ra = ra
            self.model_track_dec = dec
        else:
            self.obs_ra = ra
            self.obs_dec = dec

    def prepare_for_plot(self):
        # Remove periodicity that produces a horizontal line in the plot 
        diff_ra = np.abs(np.diff(self.model_track_ra))
        mask = np.hstack([ diff_ra > 10, [np.nan]])
        masked_data = np.ma.MaskedArray(self.model_track_ra, mask)
        self.model_track_ra = masked_data


    def fetch_observations(self,ephemerides,tmin,tmax,dt=0.02):
        # dt is in years. 0.02 years is about 7 days (a week)
        start_date = decimal_year_to_date(tmin)
        end_date = decimal_year_to_date(tmax)
        step_days = 365.0*dt

        # Build a list of datetime objects for Skyfield
        date_list = []
        time_in_years = []
        current_date = start_date
        while current_date <= end_date:
            date_list.append(current_date)
            current_date += timedelta(days=step_days)
            time_in_years.append(date_to_decimal_year(current_date))

        time = ts.utc([d.year for d in date_list], 
                      [d.month for d in date_list], 
                      [d.day for d in date_list])

        self.obs_t = time_in_years
        self.obs_x_ec = np.zeros(len(time))
        self.obs_y_ec = np.zeros(len(time))
        self.obs_z_ec = np.zeros(len(time))
        for i,t in enumerate(time):
            position = ephemerides['sun'].at(t).observe(ephemerides[self.jpl_planet_names[self.name]])
            x, y, z = position.frame_xyz(ecliptic_frame).au
            self.obs_x_ec[i] = x
            self.obs_y_ec[i] = y
            self.obs_z_ec[i] = z

In [ ]:
### SUPPORTING CODE

# Some utility functions and imports
from datetime import datetime, timedelta
from functools import partial

def decimal_year_to_date(decimal_year):
    year = int(decimal_year)
    
    fraction = decimal_year - year
    
    # Determine the total number of days in this specific year
    start_of_this_year = datetime(year, 1, 1)
    start_of_next_year = datetime(year + 1, 1, 1)
    days_in_year = (start_of_next_year - start_of_this_year).days
    
    # Calculate the exact time elapsed in seconds
    total_seconds = fraction * days_in_year * 24 * 60 * 60
    
    # Add the elapsed time to the beginning of the year
    result_date = start_of_this_year + timedelta(seconds=total_seconds)
    
    return result_date

def date_to_decimal_year(dt: datetime) -> float:
    # Get the start of the current year and the start of the next year
    start_of_this_year = datetime(dt.year, 1, 1)
    start_of_next_year = datetime(dt.year + 1, 1, 1)
    
    # Calculate the elapsed time and total duration of the year in seconds
    year_elapsed = (dt - start_of_this_year).total_seconds()
    year_duration = (start_of_next_year - start_of_this_year).total_seconds()
    
    return dt.year + (year_elapsed / year_duration)


# How the sky changes across the year


### The stars above Lehman College through the night

An animation of how the sky moves above Lehman College from sunset to dawn.


<div style="border-left: 4px solid #2196F3; padding-left: 1em; padding-top: 1em; padding-bottom: 1em; margin: 1em 0; background: #f0f8ff; color: #1a1a1a;">

**ANIMATION (work in progress):**

The point of reference will be a photograph of the "skyline" of the Bronx around Lehman college with the dark silhouettes of the surrounding buildings (or our telescope's dome, Gillet Hall, etc).
The students could control the date and how frequent the animation frames occur.
Eventually, we will replace this with photographs of the sky taken with our own telescope.
</div>


<div style="border-left: 4px solid #2196F3; padding-left: 1em; padding-top: 1em; padding-bottom: 1em; margin: 1em 0; background: #f0f8ff; color: #1a1a1a;">

**ANIMATION (work in progress):**

Exactly the same animation as above, but now the frames are taken at the same time, e.g. midnight, over a year.

The students can control the duration of this animation and how frequently the frames are drawn (although it is periodic over a year).
</div>


# Reference frames

We need some reference frame to understand the motions on objects on the sky.
As is the case in physics, a good choice of reference frame can simplify the solution of a problem.

- Horizonal (everything moves, what we see from a specific location on Earth)
- Equatorial (stars don't move*, planets move w.r.t. the stars)
- Ecliptic
- Galactic

### Let's plot the sky in Equatorial coordinates

What do we need to know for each star?

We will use a catalogue of stars from the Hipparcos satellite which observed XXX.
This catalogue includes the RA and DEC for all the stars it contains.

In [ ]:
### SUPPORTING CODE

### Get bright stars and constellations
def get_stars_constellations(mag_lim=5.0):
    # Download and load the Hipparcos catalog dataframe
    with load.open(hipparcos.URL) as f:
        df_stars = hipparcos.load_dataframe(f)

    # Separate bright stars for background display (e.g., magnitude <= 6.0)
    bright_stars = df_stars[df_stars['magnitude'] <= mag_lim]

    # Download and parse Stellarium's constellation line data
    # This returns tuples: (abbreviation, [(star_hip_id1, star_hip_id2), ...])
    url = 'https://raw.githubusercontent.com/Stellarium/stellarium/eb47095a9282cf6b981f6e37fe1ea3a3ae0fd167/skycultures/modern_st/constellationship.fab'
    with load.open(url) as f:
        constellations = stellarium.parse_constellations(f)

    # Verify both star indices exist inside our loaded Hipparcos dataframe
    cleaned_constellations = []
    for name,edges in constellations:
        flag = True
        for hip1, hip2 in edges:
            if hip1 not in bright_stars.index or hip2 not in bright_stars.index:
                flag = False
        if flag:
            cleaned_constellations.append( (name,edges) )

    return bright_stars,cleaned_constellations


def plot_stars_constellations(ax,stars,constellations):
    sizes = (7.0 - stars['magnitude']) ** 1.5
    
    ax.scatter(stars['ra_degrees'],stars['dec_degrees'],s=sizes,c='black',alpha=0.85,zorder=1)

    # Build and draw the constellation overlay lines.
    # We parse each line edge connecting hip_id1 to hip_id2.
    for name, edges in constellations:
        for hip1, hip2 in edges:
            star1 = bright_stars.loc[hip1]
            star2 = bright_stars.loc[hip2]

            # Simple handling for wrapping lines at the 0/360 boundary
            if abs(star1['ra_degrees'] - star2['ra_degrees']) > 180:
                continue 

            #if abs(star1['ra_degrees'] - star2['ra_degrees']) > np.radians(180):
            #    continue 
            
            ax.plot(
                [star1['ra_degrees'], star2['ra_degrees']],
                [star1['dec_degrees'], star2['dec_degrees']],
                color='deepskyblue', 
                linestyle='-', 
                linewidth=0.8, 
                alpha=0.6,
                zorder=2
            )

In [ ]:
### MAIN CODE

bright_stars,constellations = get_stars_constellations(mag_lim=5.0)

In [ ]:
### MAIN CODE

fig = plt.figure( figsize=(14, 5) )

ax = plt.subplot(111)
#ax = plt.subplot(111, projection='mollweide')
#bright_stars['ra_degrees'] = np.radians(bright_stars['ra_degrees'] - 180)
#bright_stars['dec_degrees'] = np.radians(bright_stars['dec_degrees'])

#ax.set_xlim(360,0)
#ax.set_ylim(-30, 30)
ax.set_xlabel("Right Ascension (Degrees)", color='black')
ax.set_ylabel("Declination (Degrees)", color='black')
ax.grid(True, linestyle=':', color='gray', alpha=0.3)

plot_stars_constellations(ax,bright_stars,constellations)

plt.tight_layout()
plt.show()

<div style="border-left: 5px solid #FFA500; background-color: rgba(255, 165, 0, 0.15); padding: 10px; border-radius: 4px; color: inherit;">

<font size=4><b>Exercise 1:<b>
    
<ol>
    <li>
        Can you locate any constellation or star that you may know? Overplot it on this figure using a red square or circle.<br>
        <i>Hint: Have a look at Matplotlib's plotting functions for a Rectangle or a Circle.</i><br>
        <i>Hint: If you cannot recognise any constellation, then you can use a stellar chart.</i>
    </li>
    <li>
        We know that RA and DEC are angles, so why we plot the sky on a square? Is there any other projection that we can use?<br>
        <i>Hint: Think of a map of the Earth and look into the 'mollweide' projection in the code above.</i>
    </li>    
    <li>
        Can you add fainter stars to this plot (or keep only the brightest ones)? Is there any other feature that you can recongize if you keep adding fainter stars?<br>
        <i>Hint: Remember, in the code above we load a catalogue of stars from the Hipparcos mission.</i>
    </li>
</ol>
</div>

# Motions of Planets (and the Sun) on the sky

### What about the ecliptic?

We will do the same plot as above but we will include the ecliptic plane.

Which celestial object(s) can be found along the ecliptic?

In [ ]:
### SUPPORTING CODE

def get_ecliptic_plane():
    # Define the ecliptic longitude (ecliptic latitude is 0 on the ecliptic)
    l = np.linspace(0,2*np.pi,500)
    obliquity = np.radians(23.439)

    ra_e = np.zeros(len(l))
    for i in range(0,len(l)):
        ra_e[i] = np.atan2( np.cos(obliquity)*np.sin(l[i]),np.cos(l[i]) )
        if l[i] > np.pi:
            ra_e[i] = ra_e[i] + 2*np.pi
    
    ra_e = np.degrees( ra_e )
    dec_e = np.degrees( np.asin(np.sin(obliquity)*np.sin(l)) )

    return ra_e,dec_e

ra_e,dec_e = get_ecliptic_plane()

In [ ]:
### MAIN CODE

fig = plt.figure( figsize=(14, 5) )

ax = plt.subplot(111)
#ax = plt.subplot(111, projection='mollweide')
#bright_stars['ra_degrees'] = np.radians(bright_stars['ra_degrees'] - 180)
#bright_stars['dec_degrees'] = np.radians(bright_stars['dec_degrees'])

ax.set_xlim(360,0)
#ax.set_ylim(-30, 30)
ax.set_xlabel("Right Ascension (Degrees)", color='black')
ax.set_ylabel("Declination (Degrees)", color='black')
ax.grid(True, linestyle=':', color='gray', alpha=0.3)

plot_stars_constellations(ax,bright_stars,constellations)

ax.plot(ra_e,dec_e,color='black',ls='--')

plt.tight_layout()
plt.show()
plt.close()

Before we examine the motions of planets on the night sky and how to describe them, let's plot the position of the Sun as a function of time.

The benefit of examining the Sun is that for all our intents and purposes we can assume it sits still in space and is not, for example, rotating around any other star or planet. We know that this is not true, but it helps us immensely in making sense of how objects in the solar system move. It is actually a pretty good assumption and inclding any of the Sun's motions (e.g. around the center of our Galaxy) has a negligible effect. Hence, the movement of the Sun against background stars is purely due to the Earth's own motion, orbiting the Sun.

In [ ]:
### SUPPORTING CODE

def plot_sky_view(planets):
    fig, ax = plt.subplots(1, 1, figsize=(14, 5))

    ax.set_xlim(360,0)
    ax.set_ylim(-30, 30)
    ax.set_xlabel("Right Ascension (Degrees)", color='black')
    ax.set_ylabel("Declination (Degrees)", color='black')
    ax.grid(True, linestyle=':', color='gray', alpha=0.15)

    ### Plot stars and constellations in the background
    plot_stars_constellations(ax,bright_stars,constellations)

    planet_paths = []
    planet_markers = []
    for p in planets:
        tmp_path, = ax.plot([], [], '-', alpha=0.3, linewidth=1, color=p.color)
        tmp_marker, = ax.plot([], [], marker=p.marker, markersize=12, label=p.name,color=p.color)
        planet_paths.append(tmp_path)
        planet_markers.append(tmp_marker)


    date_box = ax.text(0.5, 0.95, "", transform=ax.transAxes, color="black", fontsize=11, fontfamily="monospace",ha='center')
    ax.legend(handles=planet_markers,loc="upper left", framealpha=0.3)

    return fig, planet_paths, planet_markers, date_box


def update_plot_sky_view(frame,years,planets,planet_paths,planet_markers,date_box):
    for i in range(0,len(planets)):
        cur_ra = planets[i].model_track_ra[frame]
        cur_dec = planets[i].model_track_dec[frame]
        planet_paths[i].set_data(planets[i].model_track_ra[:frame], planets[i].model_track_dec[:frame])
        planet_markers[i].set_data([cur_ra], [cur_dec])

    converted_date = decimal_year_to_date(years[frame])
    date = converted_date.strftime("%Y-%B-%d")
    date_box.set_text(f"{date}")

    return (*planet_paths, *planet_markers, date_box)

In [ ]:
### MAIN CODE

t = np.linspace(0.0,1.0,50)
years = 2026.0 + t

earth = planet("earth",1,0.0167,1,0,23.439,-11.26064,114.20783,357.517)
sun = planet("sun",1,0,0,0,0,0,0,0) # Dummy arguments for the Sun, the most important is its semimajor axis being 0 which forces the Sun to stay still.

earth.get_keplerian_track(years)
earth.convert_to_ecliptic()

sun.get_keplerian_track(years)
sun.convert_to_ecliptic()
sun.convert_to_eq(earth)
sun.prepare_for_plot()


fig, planet_paths, planet_markers, date_box = plot_sky_view([sun])

ax = fig.axes[0]

#ax.plot(sun.model_track_ra,sun.model_track_dec,color=sun.color)

my_date = datetime(2026, 9, 21) # Year, month, day
my_year = date_to_decimal_year(my_date)
idx = np.abs( years - my_year ).argmin()

date_label = my_date.strftime("%Y-%B-%d")
date_box.set_text(f"{date_label}")

ax.scatter(sun.model_track_ra[idx],sun.model_track_dec[idx],color=sun.color,marker=sun.marker,s=300)


plt.tight_layout()
plt.show()

In [ ]:
### ANIMATION

# Let's animate the motion of the Sun on the sky in RA and DEC
fig_ani, planet_paths, planet_markers, date_box = plot_sky_view([sun])

# Plot the ecliptic plane
fig_ani.axes[0].plot(ra_e,dec_e,color='black',ls='--')

ani = animation.FuncAnimation(fig_ani,
                              partial(update_plot_sky_view,
                                      years=years,
                                      planets=[sun],
                                      planet_paths=planet_paths,
                                      planet_markers=planet_markers,
                                      date_box=date_box),
                              frames=len(years),
                              blit=True,
                              interval=100)
plt.close()
HTML(ani.to_html5_video())

<div style="border-left: 5px solid #FFA500; background-color: rgba(255, 165, 0, 0.15); padding: 10px; border-radius: 4px; color: inherit;">

<font size=4><b>Exercise 2:<b>
    
<ol>
    <li>
        Can you identify "special" locations in the Sun's path on the sky? Do they match with any special dates?
    </li>
    <li>
        Does the position of the Sun make sense with respect to the seasons? <br>
        <i>Bonus: Why we can completely ignore the distance to the Sun when thinking of the seasons? This will become more obvious later but you can already find the answer!</i>
    </li>
    <li>
        How does the position of the Sun relate to which constellations we can see in the night sky during summer or winter?
    </li>
    <li>
        Identify a few constellations that we can see "in the middle of the night" for each season.
    </li>
</ol>
</div>

# How planets actually move?

Define a model for the planets and plot it from a 'top-view' ('side-view' will come later, when we explain the loops, instead of just back-and-forth, of retrograde motion).

In [ ]:
### SUPPORTING CODE

import bisect

def plot_top_view(planets,flag=None):

    if flag == 'with-data':
        fig, axs = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(12,6))
        ax = axs[0]
    else:
        fig, ax = plt.subplots(1, 1, figsize=(6,6))

    # Work on the left (model) plot
    ax.set_aspect('equal')
    ax.set_xlabel("x [AU]",color='black')
    ax.set_ylabel("y [AU]",color='black')
    ax.grid(True, linestyle=':', color='gray', alpha=0.15)
    ax.scatter(0,0,color="gold",marker=(8,2,0),s=300)
    
    planet_paths = []
    planet_markers = []
    semi_majors = []
    for p in planets:
        tmp_path, = ax.plot([], [], '-', alpha=0.3, linewidth=1, color=p.color)
        tmp_marker, = ax.plot([], [], marker=p.marker, markersize=12, label=p.name,color=p.color)
        planet_paths.append(tmp_path)
        planet_markers.append(tmp_marker)
        semi_majors.append(p.a)

    max_semi = np.amax(semi_majors) + 1
    ax.set_xlim(-max_semi,max_semi)
    ax.set_ylim(-max_semi,max_semi)
    
    date_box = ax.text(0.5, 0.95, "", transform=ax.transAxes, color="black", fontsize=11, fontfamily="monospace",ha='center')
    ax.legend(handles=planet_markers,loc="upper left", framealpha=0.3)

    
    if flag == 'with-data':
        ax = axs[1]
        ax.set_aspect('equal')
        ax.set_xlabel("x [AU]",color='black')
        ax.grid(True, linestyle=':', color='gray', alpha=0.15)
        ax.scatter(0,0,color="gold",marker=(8,2,0),s=300)

        obs_date_box = ax.text(0.95, 0.05, "", transform=ax.transAxes, color="black", fontsize=11, fontfamily="monospace",ha='right')
        
        obs_paths = []
        obs_markers = []
        for p in planets:
            tmpo_path, = ax.plot([], [], marker=p.marker, markersize=6, alpha=0.4, markerfacecolor=p.color, markeredgecolor='None', linestyle='None')
            tmpo_marker, = ax.plot([], [], marker=p.marker, markersize=12, markerfacecolor=p.color, markeredgecolor='None', linestyle='None')
            obs_paths.append(tmpo_path)
            obs_markers.append(tmpo_marker)

        return fig, planet_paths, planet_markers, obs_paths, obs_markers, date_box, obs_date_box
    else:
        return fig, planet_paths, planet_markers, date_box


def update_plot_top_view(frame,years,planets,planet_paths,planet_markers,date_box):
    for i in range(0,len(planets)):
        planet_paths[i].set_data(planets[i].model_track_x_ec[:frame], planets[i].model_track_y_ec[:frame])
        planet_markers[i].set_data([planets[i].model_track_x_ec[frame]], [planets[i].model_track_y_ec[frame]])

    converted_date = decimal_year_to_date(years[frame])
    date = converted_date.strftime("%Y-%B-%d")
    date_box.set_text(f"{date}")

    return (*planet_paths, *planet_markers, date_box)

def update_plot_top_view_with_data(frame,years,planets,planet_paths,planet_markers,obs_paths,obs_markers,date_box,obs_date_box):
    for i in range(0,len(planets)):
        planet_paths[i].set_data(planets[i].model_track_x_ec[:frame], planets[i].model_track_y_ec[:frame])
        planet_markers[i].set_data([planets[i].model_track_x_ec[frame]], [planets[i].model_track_y_ec[frame]])

    converted_date = decimal_year_to_date(years[frame])
    date = converted_date.strftime("%Y-%B-%d")
    date_box.set_text(f"{date}")

    obs_date_text = []
    for i in range(0,len(planets)):
        obs_frame = bisect.bisect_left(planets[i].obs_t,years[frame]) - 1
        obs_paths[i].set_data(planets[i].obs_x_ec[:obs_frame], planets[i].obs_y_ec[:obs_frame])
        obs_markers[i].set_data([planets[i].obs_x_ec[obs_frame]], [planets[i].obs_y_ec[obs_frame]])
        converted_date = decimal_year_to_date(planets[i].obs_t[obs_frame])
        date = converted_date.strftime("%Y-%B-%d")
        obs_date_text.append(f"{planets[i].name}: {date}" )
    obs_date_box.set_text("\n".join(obs_date_text))
    
    return (*planet_paths, *planet_markers, *obs_paths, *obs_markers, date_box, obs_date_box)

In [ ]:
### MAIN CODE

t = np.linspace(0.0,3.0,200)
years2 = 2026.0 + t

earth = planet("earth",1,0.0167,1,0,23.439,-11.26064,114.20783,357.517)
mars  = planet("mars",1.8808,0.0934,1.5237,1.85,25.19,49.558,286.4623,19.412)
mercury = planet("mercury",0.241,0.2056,0.387,7.004,0.034,48.33167,29.12478,174.796)
venus = planet("venus",0.615,0.0068,0.723,3.395,177.36,76.68069,54.85229,50.115)

planets = [mercury,venus,earth,mars]
for p in planets:
    p.get_keplerian_track(years2)
    p.convert_to_ecliptic()

fig2, planet_paths2, planet_markers2, date_box2 = plot_top_view(planets)

for p in planets:
    fig2.axes[0].plot(p.model_track_x_ec,p.model_track_y_ec,color=p.color)

plt.tight_layout()
plt.show()

In [ ]:
### ANIMATION

fig_ani_2, planet_paths2, planet_markers2, date_box2 = plot_top_view(planets)
ani = animation.FuncAnimation(fig_ani_2,
                              partial(update_plot_top_view,
                                      years=years2,
                                      planets=planets,
                                      planet_paths=planet_paths2,
                                      planet_markers=planet_markers2,
                                      date_box=date_box2),
                              frames=len(years2),
                              blit=True,
                              interval=100)
plt.close()
HTML(ani.to_html5_video())

<div style="border-left: 5px solid #FFA500; background-color: rgba(255, 165, 0, 0.15); padding: 10px; border-radius: 4px; color: inherit;">

<font size=4><b>Exercise 3:<b>
    
<ol>
    <li>
        Does the above figure and animation make sense compared to what you know about the solar system?<br>
        <i>Hint: Think of the 3 Kepler's laws.</i>
    </li>
</ol>
</div>

### Let's compare this to actual data

We fetch ephemerides from JPL.

In [ ]:
### MAIN CODE

for p in planets:
    p.fetch_observations(eph,years2[0],years2[-1])

In [ ]:
### ANIMATION

fig_ani_3, planet_paths3, planet_markers3, obs_paths3, obs_markers3, date_box3, obs_date_box3 = plot_top_view(planets,flag='with-data')
ani = animation.FuncAnimation(fig_ani_3,
                              partial(update_plot_top_view_with_data,
                                      years=years2,
                                      planets=planets,
                                      planet_paths=planet_paths3,
                                      planet_markers=planet_markers3,
                                      obs_paths=obs_paths3,
                                      obs_markers=obs_markers3,
                                      date_box=date_box3,
                                      obs_date_box=obs_date_box3
                                     ),
                              frames=len(years2),
                              blit=True,
                              interval=100)
plt.close()
HTML(ani.to_html5_video())

<div style="border-left: 5px solid #FFA500; background-color: rgba(255, 165, 0, 0.15); padding: 10px; border-radius: 4px; color: inherit;">

<font size=4><b>Exercise 4:<b>

<ol>
    <li>
        What do you observe when you compare the left panel, which is actually a theoretical model of the planet motions, with the observations in the right? There is clearly good agreement between the model and the data, but can you spot any discrepancies?<br>
        <i>Hint: Paying attention to planet <a href="https://en.wikipedia.org/wiki/Syzygy_(astronomy)" target="_blank">syzygies</a> or <a href="https://en.wikipedia.org/wiki/Conjunction_(astronomy)" target="blank">conjuctions</a> will help.</i>
    </li>
</ol>
</div>

# Models vs Data

A big part of what scientists have been doing over the centuries is comparing theories and models to data obtained by experiments.

Here we will do such a comparison, first "by-eye".

In [ ]:
### MAIN CODE

for p in planets:
    p.get_circular_track(years2)
    p.convert_to_ecliptic()

In [ ]:
### ANIMATION

fig_ani_4, planet_paths4, planet_markers4, obs_paths4, obs_markers4, date_box4, obs_date_box4 = plot_top_view(planets,flag='with-data')
ani = animation.FuncAnimation(fig_ani_4,
                              partial(update_plot_top_view_with_data,
                                      years=years2,
                                      planets=planets,
                                      planet_paths=planet_paths4,
                                      planet_markers=planet_markers4,
                                      obs_paths=obs_paths4,
                                      obs_markers=obs_markers4,
                                      date_box=date_box4,
                                      obs_date_box=obs_date_box4
                                     ),
                              frames=len(years2),
                              blit=True,
                              interval=100)
plt.close()
HTML(ani.to_html5_video())

<div style="border-left: 5px solid #FFA500; background-color: rgba(255, 165, 0, 0.15); padding: 10px; border-radius: 4px; color: inherit;">

<font size=4><b>Exercise 5:<b>
<font size=3>
    
In this case, the left panel is a different model - circular instead of elliptical - while the right panel is still the data, i.e. what we observe in nature. 
<ol>
    <li>
        Is this model better than the previous one? <br>
        <i>Hint: Look at the periods of the planets, but also at random moments during their orbits. <i>
    </li>
    <li>
        Can any discrepancies that you found be explained by the "lag" of the observations as was the case in Exercise 4?
    </li>
</ol>
</div>

### View from the Earth

The data in our experiments, what is actually being measured, are the positions of the planets on the celestial sphere with respect to the background stars and our reference frame.
It is possible to obtain a measurement of the distance of a planet from Earth if, for example, we use radar instruments, but we can ignore this type of measurements for now.
The "top-view" of the solar system is somewhat misleading in this case, so let's switch to the view from the Earth and let's use equatorial coordinates, as we have done above.

In [ ]:
### MAIN CODE

t = np.linspace(0.0,1.6,100)
years3 = 2026.0 + t

earth.get_keplerian_track(years3)
earth.convert_to_ecliptic()

mars.get_keplerian_track(years3)
#mars.get_circular_track(years3)
mars.convert_to_ecliptic()
mars.convert_to_eq(earth)
mars.prepare_for_plot()

In [ ]:
### ANIMATION

fig_ani_5, planet_paths5, planet_markers5, date_box5 = plot_sky_view([mars])
ani = animation.FuncAnimation(fig_ani_5,
                              partial(update_plot_sky_view,
                                      years=years3,
                                      planets=[mars],
                                      planet_paths=planet_paths5,
                                      planet_markers=planet_markers5,
                                      date_box=date_box5),
                              frames=len(years3),
                              blit=True,
                              interval=100)
plt.close()
HTML(ani.to_html5_video())

<div style="border-left: 5px solid #FFA500; background-color: rgba(255, 165, 0, 0.15); padding: 10px; border-radius: 4px; color: inherit;">

<font size=4><b>Exercise 6:<b>

There is a lot to unpack here!

<ol>
    <li>
        First, let's ignore that strange loop for the moment. What do you observe when you compare the orbit of Mars to the "apparent orbit" of the Sun?<br>
        ("apparent orbit" is in quotes because it is the Earth moving around the Sun and not the other way around)<br>
        <i>Hint: Remember the definition of the ecliptic and find above the piece of code that can be used to plot it here.<i>
    </li>
    <li>
        How can these loops be explained? Can you see them in the orbits of other planets?<br>
        <i>Hint: Imagine that you are on Earth (the blue dot) on the "top-view" animations above looking outwards (that would be on the plane of the animation) towards the other planets and the background stars.</i>
    </li>
    <li>
        Why this "backwards" motion has the shape of a small loop, or an "S", as opposed to being on a flat line?
    </li>
    <li>
         Are these shapes random? Under which conditions can we get the same, or a very similar shape of such a loop? What about other planets?
    </li>
    <li>
        Going back to Exercise 2, how can you make the Sun move in a similar way on the sky as seen from Earth? What would this mean for the Sun?
    </li>
</ol>
</div>

### Full 3-dimensional orbits

Explain the full 3-dimensional orbits of the planets in the solar system.
Introduce and explain with figures the 6 parameters needed to define an orbit, either position and velocity vectors, or the orbital elements (strong link to mechanics).
Explain the inclination that leads to the 2D shape of the loops or "S" shapes.

*Is the Solar System flat?*

Let's plot the orbit inclinations of all the planets looking at the solar system "edge-on" ("side-view", or *x-z*, or *y-z* planes) as opposed to the "face-on" ("top-view", *x-y* plane) that we have seen above.

In [ ]:
### MAIN CODE

earth = planet("earth",1,0.0167,1,0,23.439,-11.26064,114.20783,357.517)
mars  = planet("mars",1.8808,0.0934,1.5237,1.85,25.19,49.558,286.4623,19.412)
jupiter = planet("jupiter",11.862,0.0487,5.204,1.304,3.13,100.556,-85.8023,19.67)
saturn = planet("saturn",29.447,0.052,9.573,2.486,26.73,113.71504,-21.2831,317.020)
mercury = planet("mercury",0.241,0.2056,0.387,7.004,0.034,48.33167,29.12478,174.796)
venus = planet("venus",0.615,0.0068,0.723,3.395,177.36,76.68069,54.85229,50.115)
uranus = planet("uranus",83.749,0.0469,19.165,0.77,97.77,74.22988,96.73436,142.2679)
neptune = planet("neptune",164.79,0.0097,30.178,1.77,28.32,131.72169,-86.75034,259.883)

all_planets = [mercury,venus,earth,mars,jupiter,saturn,uranus,neptune]

In [ ]:
### MAIN CODE

fig = plt.figure( figsize=(8, 2) )
ax = plt.subplot(111)

for p in all_planets:
    ax.axline((0,0),slope=np.radians(p.i),color=p.color)

ax.set_aspect('equal')
ax.set_xlabel("x (arbitrary units)", color='black')
ax.set_ylabel("z (arbitrary units)", color='black')
ax.set_xlim(-0.20,0.20)
ax.set_ylim(-0.02,0.02)

plt.tight_layout()
plt.show()

Note that the y-axis range above is 10 times smaller than the x-axis. The solar system is pretty flat.

We have used arbitrary units on the x and y axes, which is as if we extended the orbit of each planet to infinity. In reality, each line should extend by as much as the planet's semi-major axis. For example, Mercury, which has the most inclined orbit with respect to the other planets, has the smallest orbit size.

<div style="border-left: 5px solid #FFA500; background-color: rgba(255, 165, 0, 0.15); padding: 10px; border-radius: 4px; color: inherit;">

<font size=4><b>Bonus Exercise 1:</b>

<ol>
    <li>
        Can you reproduce the above plot of the planet orbit inclinations including the size of each planet's orbit?<br>
        <i>Hint: The units of the axes should be changed to AU.</i>
    </li>
</ol>

</div>

<div style="border-left: 5px solid #FFA500; background-color: rgba(255, 165, 0, 0.15); padding: 10px; border-radius: 4px; color: inherit;">

<font size=4><b>Bonus Exercise 2:</b>

<i>
    <b>WORK IN PROGRESS</b>

Explain the seasons and their irrelevance to the position on the orbit of the Earth.
Bonus Problem: Calculate how much energy a location on Earth at fixed latitude (either north or south but has to be the same) receives in **day** in the winter as opposed to the summer. In a single second, the location on the orbit closest to the Sun receives the most energy, but the inclination angle makes days shorter or longer, and this is the determining factor on how much total energy is received. If the Earth's orbit was very elliptical, this would be quite different.</i>
</div>

<div style="border-left: 5px solid #FFA500; background-color: rgba(255, 165, 0, 0.15); padding: 10px; border-radius: 4px; color: inherit;">

<font size=4><b>Exercise 7:<b>

<ol>
    <li>
        Which parameter of Mars' orbit you should change to make the loops or "S" shapes flat, or in other words, constraining the motion of Mars on a single line on the sky.
    </li>
    <li>
        Do you think that a Martian astronomer could observe the Earth doing something similar on the martian night sky? Perform such an experiment.<br>
        <i>Hint: You can assume that all the background stars, and therefore all the constellations, are exactly the same when seen from Mars as seen from the Earth (which is true for the vast majority of stars).</i>
    </li>
</ol>
</div>

### Which model agrees with the data?

We will now plot the model and the data, both as seen from Earth. Our goal now is to have a model that matches exactly the observations, not only in the shape of the orbit but also on specific dates. This last part is crucial.

<div style="border-left: 4px solid #e57373; padding-left: 1em; padding-top: 1em; padding-bottom: 1em; margin: 1em 0; background: #fdecea; color: #1a1a1a;">

**WARNING:** 

Because the Longitude of Ascending Node and the Argument of Periapsis (other parameters too, but mostly these two) change over time (due to apsidal precession), I have to run a small optimization routine to get the best-fit parameters for some specific time range, e.g. 2026-2027.

I could use the approx. relations of how these quantities evolve with time, but because I don't know exactly what Skyfield is using to provide the "data", the safest and quickest option is just to optimize.

I could then ask the students to select another time period far in the past or future to see that the model breaks because of all the perturbations to Earth's orbit.

</div>

In [ ]:
### MAIN CODE

tmp_earth = planet("earth",1,0.0167,1,0,23.439,0,114.20783,357.517)
tmp_mars  = planet("mars",1.8808,0.0934,1.5237,1.85,25.19,49.5,286.2,19.412)
#mars  = planet("mars",1.8808,0.0934,1.5237,1.85,25.19,3.37368,333.018413,19.412)
#earth = planet("earth",1,0.01704,1,0,23.436,0.00018,101.7419,357.517)


my_planet = tmp_mars
ref_planet = tmp_earth

# Define a date range
year_min, year_max, N = (2026.6, 2027.6, 40)
#year_min, year_max, N = (2016.1, 2017.1, 40)

# First, get observations for Mars in the selected time range
# We want to evaluate the model at *exactly* the same time as the observations
dt = float( (year_max - year_min)/N )
my_planet.fetch_observations(eph,year_min,year_max,dt=dt)
years4 = np.array(my_planet.obs_t)

# For a Keplerian model
ref_planet.get_keplerian_track(years4)
ref_planet.convert_to_ecliptic()

my_planet.get_keplerian_track(years4)
#mars.get_circular_track(years4)
my_planet.convert_to_ecliptic()
my_planet.convert_to_eq(ref_planet) # Convert model to RA,DEC
my_planet.convert_to_eq(ref_planet,flag='obs') # Convert observations to RA,DEC




### Plot the model and data
fig, ax = plt.subplots(1, 1, figsize=(14, 5))

### Plot stars and constellations in the background
bright_stars,cleaned_constellations = get_stars_constellations(mag_lim=5.0)
plot_stars_constellations(ax,bright_stars,constellations)

# Plot the observations and model
ax.scatter(my_planet.obs_ra, my_planet.obs_dec, marker='o', ec=my_planet.color, fc='None',label="Data")
ax.scatter(my_planet.model_track_ra, my_planet.model_track_dec, marker='^', ec='None', fc=p.color,label="Model")

# Set the bounds of the plot based on the observations
max_ra = max(max(my_planet.obs_ra),max(my_planet.model_track_ra))
min_ra = min(min(my_planet.obs_ra),min(my_planet.model_track_ra))
offset_ra = 0.3*(max_ra - min_ra) # 20% of the range
max_dec = max(max(my_planet.obs_dec),max(my_planet.model_track_dec))
min_dec = min(min(my_planet.obs_dec),min(my_planet.model_track_dec))
offset_dec = 0.3*(max_dec - min_dec) # 20% of the range
ax.set_xlim(max_ra+offset_ra,min_ra-offset_ra)
ax.set_ylim(min_dec-offset_dec,max_dec+offset_dec)
#ax.set_xlim(360,0)

# Decorate the plot
ax.set_xlabel("Right Ascension (Degrees)", color='black')
ax.set_ylabel("Declination (Degrees)", color='black')
ax.grid(True, linestyle=':', color='gray', alpha=0.15)
ax.legend()

plt.show()
plt.close()

<div style="border-left: 5px solid #FFA500; background-color: rgba(255, 165, 0, 0.15); padding: 10px; border-radius: 4px; color: inherit;">

<font size=4><b>Exercise 8:<b>

<ol>
    <li>
        Which model, the Keplerian (elliptical orbits) or the Copernician (circular orbits), agrees better with the data?
    </li>
    <li>
        What about other planets?
    </li>
    <li>
        What do you observe if you select other time periods in the past or future and perform the same comparison?<br>
        <i>Hint: Move progressively away from the year 2026, first by a few years, then by a decade, and so on.</i>
    </li>
</ol>
</div>

<div style="border-left: 4px solid #2196F3; padding-left: 1em; padding-top: 1em; padding-bottom: 1em; margin: 1em 0; background: #f0f8ff; color: #1a1a1a;">

<font size=4><b>So everything moves?</b>

<i>
    <b>WORK IN PROGRESS</b>

There seems to be no constant reference point in the Solar System and everything moves with respect to each other!
How could we then study the motions of planets and other celestial bodies in our neighbourhood?
Actually, fundamental laws of physics come to the rescue.
Things like conservation of energy and angular momentum.
This is the topic of the field of physics called celestial mechanics, and mechanics in general.
</i>
</div>

### Epicycles - *Work in Progress*

A completely different theory, the geocentric ptolemaic version of the solar system, which presents a competitive model to the Keplerian and Copernician (circular) ones.



<div style="border-left: 5px solid #FFA500; background-color: rgba(255, 165, 0, 0.15); padding: 10px; border-radius: 4px; color: inherit;">

<font size=4><b>Exercise 9:<b>

<ul>
    <li>
        BLA BLA
    </li>
</ul>
</div>

In [ ]:
# EOF